## Catalog, Schema and Volume Creation Or Reusing

In [0]:
%sql
use catalog end_to_end_projects;

In [0]:
%sql
create schema if not exists sports_equipments_sales;

In [0]:
%sql
use schema sports_equipments_sales;

In [0]:
%sql
create volume if not exists incoming_files;

## Creation of Folders [Source, Bronze, Silver, Gold]

In [0]:
base_path = "/Volumes/end_to_end_projects/sports_equipments_sales/incoming_files/"

source_path = f"{base_path}/source"
bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"


In [0]:
dbutils.fs.mkdirs(source_path)

In [0]:
dbutils.fs.mkdirs(bronze_path)
dbutils.fs.mkdirs(silver_path)
dbutils.fs.mkdirs(gold_path)

In [0]:
dbutils.widgets.text("full_load", "","File Name")

In [0]:
file_name = dbutils.widgets.get("full_load")
print(f"Selected File: {file_name}")

In [0]:
display(dbutils.fs.ls(source_path))

## Reading the Data from Source File

In [0]:
full_load = f"{source_path}/{file_name}"
print(f"Full load :{full_load}")

In [0]:
df_source = spark.read.csv(full_load, header = True, inferSchema = True)
display(df_source.limit(10))
print(f"Count Of Records: {df_source.count()}")
df_source.printSchema()

In [0]:
from pyspark.sql.functions import *

### Valdating the Source Data Before Writing into the Bronze


In [0]:
null_id_count = df_source.filter(col("sale_id").isNull()).count()
duplicate_id_count = df_source.groupBy("sale_id").count().filter(col("count") > 1)
print(f"Null ID Count: {null_id_count}")
print(f"Duplicate ID Count: {duplicate_id_count.count()}")

In [0]:
df_bronze = df_source.withColumn("ingestion_timestamp", current_timestamp())
df_bronze.printSchema()

### Writing Data into Bronze Delta Table

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable("sports_equipment_sales_bronze")

### Reading the Bronze Delta Table

In [0]:
df_bronze = spark.table("sports_equipment_sales_bronze")
display(df_bronze.limit(10))
print(f"Count of Records in Bronze Delta Table: {df_bronze.count()}")
df_bronze.printSchema()

## Validating the Bronze Data By Using the Business Rules

In [0]:
null_sale_id_count = df_bronze.filter(col("sale_id").isNull()).count()
duplicate_sale_id_count = df_bronze.groupBy("sale_id").count().filter(col("count") > 1)
null_customer_id_count = df_bronze.filter(col("customer_id").isNull()).count()
null_product_id_count = df_bronze.filter(col("product_id").isNull()).count()
null_store_id_count = df_bronze.filter(col("store_id").isNull()).count()
null_salesperson_id_count = df_bronze.filter(col("salesperson_id").isNull()).count()
null_sale_date_count = df_bronze.filter(col("sale_date").isNull()).count()
null_last_updated_count = df_bronze.filter(col("last_updated").isNull()).count()
invalid_quantity_count = df_bronze.filter(col("quantity") <= 0).count()
invalid_unit_price_count = df_bronze.filter(col("unit_price") <= 0).count()
invalid_discount_amount_count = df_bronze.filter(col("discount_amount") < 0).count()
invalid_tax_amount_count = df_bronze.filter(col("tax_amount") < 0).count()
invalid_final_amount_count = df_bronze.filter(col("final_amount") <= 0).count()
invalid_date_check_count = df_bronze.filter(col("sale_date") > to_date(col("last_updated"))).count()
invalid_sale_status_count = df_bronze.filter(col("sale_status").isNull() |~col("sale_status").isin("COMPLETED", "RETURNED", "CANCELLED")).count()

print(f"Null Sale ID Count: {null_sale_id_count}")
print(f"Duplicate Sale ID Count: {duplicate_sale_id_count.count()}")
print(f"Null Customer ID Count: {null_customer_id_count}")
print(f"Null Product ID Count: {null_product_id_count}")
print(f"Null Store ID Count: {null_store_id_count}")
print(f"Null Sales Person ID Count: {null_salesperson_id_count}")
print(f"Null Sale Date Count: {null_sale_date_count}")
print(f"Null Last Updated Count: {null_last_updated_count}")
print(f"Invalid Quantity Count: {invalid_quantity_count}")
print(f"Invalid Unit Price Count: {invalid_unit_price_count}")
print(f"Invalid Discount Amount Count: {invalid_discount_amount_count}")
print(f"Invalid Tax Amount Count: {invalid_tax_amount_count}")
print(f"Invalid Final Amount Count: {invalid_final_amount_count}")
print(f"Invalid Date Check Count: {invalid_date_check_count}")
print(f"Invalid Sale Status Count: {invalid_sale_status_count}")


### Apply the Required Silver Transformations

In [0]:
from pyspark.sql.window import *

In [0]:
df_silver = df_bronze.filter(col("sale_id").isNotNull())\
    .filter(col("customer_id").isNotNull())\
        .filter(col("product_id").isNotNull())\
            .filter(col("store_id").isNotNull())\
                .filter(col("salesperson_id").isNotNull())\
                    .filter(col("sale_date").isNotNull())\
                        .filter(col("last_updated").isNotNull())\
                            .filter(col("quantity") > 0)\
                                .filter(col("unit_price") > 0)\
                                    .filter(col("discount_amount") >= 0)\
                                        .filter(col("tax_amount") >= 0)\
                                            .filter(col("final_amount") > 0)\
                                                .filter(col("sale_date") <= to_date(col("last_updated")))\
                                                    .filter((col("sale_status").isNotNull()) & (col("sale_status").isin("COMPLETED", "RETURNED","CANCELLED")))\
                                                        .withColumn("row_num", row_number().over(Window.partitionBy("sale_id").orderBy(col("last_updated").desc())))\
                                                            .filter(col("row_num") == 1).drop("row_num")
display(df_silver.limit(5))
df_silver.printSchema()
print(f"Count of Records in Silver Table: {df_silver.count()}")

### Writing Data into the Silver Delta Table

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("sports_equipment_sales_silver")

## Read and Verify the Silver Delta Table

In [0]:
df_silver_delta = spark.table("sports_equipment_sales_silver")
display(df_silver_delta.limit(5))
print(f"Count of Records in Silver Delta Table: {df_silver_delta.count()}")
df_silver_delta.printSchema()